In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 28. Data text (Data Parallel)

> **Learning Goals**
> - Data text text text(All-Reduce)text text
> - text text text text
> - DDP (DistributedDataParallel)text text text

## 28.1 text GPUtext text

LLM Trainingtext text·text text:
- 7B Model FP16 + AdamW = ~70GB
- text text = text text text
- Speed: 1 GPUtext text

text: text GPUtext text.

## 28.2 Data text text text

1. text Model text $N$text GPUtext text
2. Datatext $N$text text GPUtext text
3. text GPUtext text Datatext text Calculation
4. text text text → text GPUtext text
5. text GPUtext text text text → Model text text

text:
- text text: $\mathbf{g}_i = \nabla \mathcal{L}_i(\theta)$
- text: $\mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{g}_i$
- text: $\theta \leftarrow \theta - \eta \mathbf{g}$

text text text: $B_{\text{eff}} = N \cdot B_{\text{local}}$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Data text text (text GPUtext gradient accumulationtext text)
class DataParallelSimulator:
    """text GPUtext N-GPU DDPtext text text."""
    def __init__(self, model, n_gpus=4):
        self.model = model
        self.n_gpus = n_gpus
        # Ntext "text GPU" = text Modeltext text (gradient accumulationtext text)
        # text text Modeltext gradienttext n_gpustext text
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    def step(self, x_batch, y_batch):
        """x_batch: (N*B, ...), Ntext GPU text Btext Sample."""
        # 1. Batchtext Ntext
        chunks = torch.chunk(x_batch, self.n_gpus)
        y_chunks = torch.chunk(y_batch, self.n_gpus)

        # 2. text "GPU"text Gradient Computation (gradient accumulation)
        self.optimizer.zero_grad()
        total_loss = 0
        for i in range(self.n_gpus):
            out = self.model(chunks[i])
            loss = F.mse_loss(out, y_chunks[i]) / self.n_gpus  # Mean
            loss.backward()  # Gradient text
            total_loss += loss.item() * self.n_gpus

        # 3. optimizer.step()text text Gradienttext (text Meantext)
        self.optimizer.step()
        return total_loss / self.n_gpus

# text Modeltext text
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 20)
)

dp = DataParallelSimulator(model, n_gpus=4)

# Comparison: text Batch vs DDP text
x = torch.randn(64, 20)
y = torch.randn(64, 20)

loss = dp.step(x, y)
print(f"DDP text 1 step loss: {loss:.4f}")
print(f"n_gpus=4, batch=64 → text GPU batch=16")
print(f"Effecttext Batch Magnitude: {dp.n_gpus * 16} = 64")


## 28.3 All-Reduce text

text GPUtext text text text GPUtext text text text:
- **Ring All-Reduce**: $2(N-1)$ text, text text $|\theta|/N$ text
- **Tree All-Reduce**: text text

text: $O(|\theta|)$ per step. 7B Modeltext text text 14GB text.

text text text → text-text text text.


In [ ]:
# All-Reduce text
def all_reduce_sum(grads_list):
    """text GPUtext Gradienttext Sumtext text GPUtext text."""
    # Sumtext
    total = sum(grads_list)
    # text GPUtext text Value text
    return [total.clone() for _ in grads_list]

# 4text GPU text
torch.manual_seed(0)
n_gpus = 4
local_grads = [torch.randn(5) for _ in range(n_gpus)]
print("text GPU text text:")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g.numpy().round(3)}")

# All-Reduce (Mean)
reduced = all_reduce_sum(local_grads)
avg = [r / n_gpus for r in reduced]
print("\nAll-Reduce text (text):")
for i, g in enumerate(avg):
    print(f"  GPU {i}: {g.numpy().round(3)}")
print("\n=> text GPUtext text Mean Gradienttext text.")


## 28.4 PyTorch DDP (text)

text PyTorch text:
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl', rank=rank, world_size=world_size)
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# text model(x)text text text text
```

Colab text GPUtext text text GPU DDPtext text text text, gradient accumulationtext text text text text.


In [ ]:
# text GPU DDP text
import time
from llm_math.bench import time_fn

# text Modeltext DDP vs text text Comparison
def make_model(d=512):
    return nn.Sequential(
        nn.Linear(d, d * 2), nn.ReLU(),
        nn.Linear(d * 2, d), nn.ReLU(),
        nn.Linear(d, d)
    )

# text text text
torch.manual_seed(0)
model_single = make_model()
opt_single = torch.optim.AdamW(model_single.parameters(), lr=1e-3)

def step_single(x, y):
    opt_single.zero_grad()
    out = model_single(x)
    loss = F.mse_loss(out, y)
    loss.backward()
    opt_single.step()
    return loss

# DDP text (4 GPU, text 1/4 text)
torch.manual_seed(0)
model_ddp = make_model()
opt_ddp = torch.optim.AdamW(model_ddp.parameters(), lr=1e-3)

def step_ddp(x, y, n_gpus=4):
    opt_ddp.zero_grad()
    x_chunks = torch.chunk(x, n_gpus)
    y_chunks = torch.chunk(y, n_gpus)
    total_loss = 0
    for xc, yc in zip(x_chunks, y_chunks):
        out = model_ddp(xc)
        loss = F.mse_loss(out, yc) / n_gpus  # text
        loss.backward()  # text text
        total_loss += loss.item() * n_gpus
    opt_ddp.step()
    return total_loss / n_gpus

# Time Comparison (text text text text)
d = 512
batch_size = 128
x = torch.randn(batch_size, d)
y = torch.randn(batch_size, d)

t_single = time_fn(step_single, x, y, device='cpu', warmup=2, repeat=5)
t_ddp = time_fn(step_ddp, x, y, device='cpu', warmup=2, repeat=5)
print(f"text text Batch (B=128): {t_single['mean_ms']:.3f} ms")
print(f"DDP text (4 GPU, text B=32): {t_ddp['mean_ms']:.3f} ms")
print("\n=> text GPUtext DDP text text text (sequential text).")
print("   text text GPUtext text text text text.")


## 28.5 Large Batch Training — LARS, LAMB

text text Training text. text Trainingtext text text:

**LARS** (Layer-wise Adaptive Rate Scaling):
$$\eta_\ell = \eta \cdot \frac{\|\theta_\ell\|}{\|\nabla \mathcal{L}_\ell\| + \lambda \|\theta_\ell\|}$$

**LAMB**: LARS + Adam. LLM text Trainingtext text.

## 28.6 text-text text

Backpropagationtext text text. text text text Calculationtext **text All-Reduce text**, text text Backpropagationtext text.

text text Timetext text text text text.

## 28.7 Key Takeaways

| text | text |
|---|---|
| Data text | text Model Ntext, Data text |
| All-Reduce | text text |
| text text | $N \cdot B_{\text{local}}$ |
| text | $O(\|\theta\|)$ per step |
| LARS/LAMB | text text Training text |

## Exercises

1. 4-GPU DDP text text GPUtext text Datatext Trainingtext text text text text.
2. text text text 32, 64, 128, 256text Trainingtext text Comparisontext.
3. All-Reducetext text $O(\|\theta\|)$text text text.
4. text-text text text Speedtext text text.
5. LARStext text text text degreestext text text.

> Solutions: `solutions/ch28_solutions.ipynb`
